# Data Cleaning & Preparation Notebook: Potato Quality Classification Fresh vs Rotten

---
## 0. Setup & Configuration

In [ ]:
import os
import shutil
import hashlib
from pathlib import Path
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_ROOT = Path(".")

SOURCES = {
    "kaggle_dataset_1": {
        "fresh_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_1" / "Fruits_Vegetables_Dataset" / "Vegetables" / "FreshPotato",
        ],
        "rotten_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_1" / "Fruits_Vegetables_Dataset" / "Vegetables" / "RottenPotato",
        ],
        "description": "Web-scraped images with varied backgrounds and lighting conditions",
        "url": "https://www.kaggle.com/datasets/swoyam2609/fresh-and-stale-classification",
    },
    "kaggle_dataset_2": {
        "fresh_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Train" / "FreshPotato",
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Test" / "FreshPotato",
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Validation" / "FreshPotato",
        ],
        "rotten_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Train" / "RottenPotato",
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Test" / "RottenPotato",
            DATA_ROOT / "raw" / "kaggle_dataset_2" / "Dataset" / "Visual_Dataset" / "Validation" / "RottenPotato",
        ],
        "description": "Studio-style images with consistent gray background and controlled lighting",
        "url": "https://www.kaggle.com/datasets/muhriddinmuxiddinov/fruits-and-vegetables-dataset/data",
    },
    "kaggle_dataset_3": {
        "fresh_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_3" / "dataset" / "Train" / "freshpotato",
            DATA_ROOT / "raw" / "kaggle_dataset_3" / "dataset" / "Test" / "freshpatato",
        ],
        "rotten_dirs": [
            DATA_ROOT / "raw" / "kaggle_dataset_3" / "dataset" / "Train" / "rottenpotato",
            DATA_ROOT / "raw" / "kaggle_dataset_3" / "dataset" / "Test" / "rottenpatato",
        ],
        "description": "Mixed images with white/varied backgrounds, resized with padding",
        "url": "https://www.kaggle.com/datasets/filipemonteir/fresh-and-rotten-fruits-and-vegetables",
    },
}

STAGING_DIR = DATA_ROOT / "_staging"
PROCESSED_DIR = DATA_ROOT / "processed"
METADATA_DIR = DATA_ROOT / "metadata"

OUTPUT_FORMAT = ".jpg"
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 0.10

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

print("Configuration loaded.")
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Split ratios: Train {TRAIN_RATIO:.0%} / Val {VAL_RATIO:.0%} / Test {TEST_RATIO:.0%}")

## 1. Inventory Scan All Raw Datasets
Before any processing, I first check which folders exist, count images per source, and identify file formats.

In [ ]:
def get_image_files(folder):
    """Return sorted list of image file paths in a folder."""
    if not folder.exists():
        return []
    return sorted([f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS])

summary_rows = []
total_found = 0
missing_dirs = []

for source_name, source_info in SOURCES.items():
    for label, dir_key in [("fresh", "fresh_dirs"), ("rotten", "rotten_dirs")]:
        for folder in source_info[dir_key]:
            files = get_image_files(folder)
            formats = {}
            for f in files:
                ext = f.suffix.lower()
                formats[ext] = formats.get(ext, 0) + 1
            exists = folder.exists()
            if not exists:
                missing_dirs.append(str(folder))
            summary_rows.append({
                "source": source_name, "label": label,
                "folder": str(folder.relative_to(DATA_ROOT)),
                "exists": "yes" if exists else "NO",
                "images": len(files), "formats": str(formats) if formats else "-",
            })
            total_found += len(files)

summary_df = pd.DataFrame(summary_rows)
print("=" * 90)
print("RAW DATASET INVENTORY")
print("=" * 90)
print(summary_df.to_string(index=False))
print(f"\nTotal images found across all sources: {total_found}")

## 2. Format Standardization
All images are converted to a single format (JPEG) to ensure consistency. This step also:
- Converts RGBA/P/LA images to RGB (removes alpha channels from PNGs)
- Converts .webp and other formats to .jpg
- Detects and logs corrupt or unreadable images

In [ ]:
def convert_to_jpg(src_path, dst_path):
    """Convert an image to RGB JPEG format."""
    try:
        img = Image.open(src_path)
        if img.mode != "RGB":
            img = img.convert("RGB")
        img.save(dst_path, "JPEG", quality=95)
        return True
    except Exception as e:
        print(f"  ⚠ SKIPPED (corrupt/unreadable): {src_path.name} — {e}")
        return False

import time
if STAGING_DIR.exists():
    for attempt in range(3):
        try:
            shutil.rmtree(STAGING_DIR)
            break
        except PermissionError:
            print(f"  Waiting for OneDrive to release files (attempt {attempt+1}/3)...")
            time.sleep(3)

metadata_records = []
skipped_files = []

for source_name, source_info in SOURCES.items():
    for label, dir_key in [("fresh", "fresh_dirs"), ("rotten", "rotten_dirs")]:
        out_dir = STAGING_DIR / label
        out_dir.mkdir(parents=True, exist_ok=True)
        for folder in source_info[dir_key]:
            files = get_image_files(folder)
            if not files:
                continue
            subfolder = str(folder.relative_to(DATA_ROOT / "raw"))
            print(f"Processing: {subfolder} ({len(files)} images)")
            converted = 0
            for f in files:
                new_name = f"{source_name}__{f.stem}{OUTPUT_FORMAT}"
                dst = out_dir / new_name
                counter = 1
                while dst.exists():
                    new_name = f"{source_name}__{f.stem}_{counter}{OUTPUT_FORMAT}"
                    dst = out_dir / new_name
                    counter += 1
                if convert_to_jpg(f, dst):
                    converted += 1
                    metadata_records.append({
                        "staging_name": new_name, "original_name": f.name,
                        "original_format": f.suffix.lower(),
                        "source_dataset": source_name, "source_path": subfolder,
                        "label": label,
                    })
                else:
                    skipped_files.append({"file": f.name, "source": source_name, "label": label})
            print(f"  → Converted: {converted}/{len(files)}")

print(f"\n{'=' * 50}")
print(f"Successfully converted: {len(metadata_records)}")
print(f"Skipped (corrupt):      {len(skipped_files)}")
if skipped_files:
    print(pd.DataFrame(skipped_files).to_string(index=False))

---
## 3. Duplicate Detection & Removal

In [ ]:
def compute_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

print("Computing MD5 hashes for all images...")
hash_records = []
for label in ["fresh", "rotten"]:
    folder = STAGING_DIR / label
    for f in sorted(f for f in folder.iterdir() if f.suffix.lower() == OUTPUT_FORMAT):
        hash_records.append({
            "file": f.name, "path": str(f), "label": label,
            "md5": compute_md5(f),
        })
hash_df = pd.DataFrame(hash_records)

# Find and remove exact duplicates
to_remove = hash_df[hash_df.duplicated(subset=["md5", "label"], keep="first")]
print(f"\nFound {len(to_remove)} exact duplicates — removing them now...\n")

for _, row in to_remove.iterrows():
    Path(row["path"]).unlink()

removed_names = set(to_remove["file"])
metadata_records = [r for r in metadata_records if r["staging_name"] not in removed_names]

for label in ["fresh", "rotten"]:
    count = sum(1 for r in metadata_records if r["label"] == label)
    print(f"  {label}: {count} images remaining")
print(f"  Total: {len(metadata_records)} images")

## 4. Uniform Renaming
All images are renamed to: `fresh_0001.jpg`, `fresh_0002.jpg`, `rotten_0001.jpg`, etc.

In [ ]:
old_to_new = {}

for label in ["fresh", "rotten"]:
    folder = STAGING_DIR / label
    files = sorted(f for f in folder.iterdir() if f.suffix.lower() == OUTPUT_FORMAT)
    
    temp_paths = []
    for idx, f in enumerate(files, start=1):
        new_name = f"{label}_{idx:04d}{OUTPUT_FORMAT}"
        temp_name = f"__temp__{new_name}"
        temp_path = folder / temp_name
        f.rename(temp_path)
        temp_paths.append((temp_path, new_name, f.name))
    
    for temp_path, new_name, old_name in temp_paths:
        final_path = folder / new_name
        temp_path.rename(final_path)
        old_to_new[old_name] = new_name

for record in metadata_records:
    record["final_name"] = old_to_new.get(record["staging_name"], record["staging_name"])

for label in ["fresh", "rotten"]:
    count = len(list((STAGING_DIR / label).glob(f"*{OUTPUT_FORMAT}")))
    print(f"{label}: {count} images renamed")
print(f"\nTotal: {len(old_to_new)} images with clean names")

---
## 5. Train / Validation / Test Split
Scikit-learn's `train_test_split` with 70/20/10 ratio. Split per class for balanced representation.

In [ ]:
import time

if PROCESSED_DIR.exists():
    for attempt in range(3):
        try:
            shutil.rmtree(PROCESSED_DIR)
            break
        except PermissionError:
            print(f"  Waiting for OneDrive to release files (attempt {attempt+1}/3)...")
            time.sleep(3)

for split in ["train", "val", "test"]:
    for label in ["fresh", "rotten"]:
        (PROCESSED_DIR / split / label).mkdir(parents=True, exist_ok=True)

split_summary = []
for label in ["fresh", "rotten"]:
    folder = STAGING_DIR / label
    files = sorted(f for f in folder.iterdir() if f.suffix.lower() == OUTPUT_FORMAT)
    train_files, remaining = train_test_split(files, test_size=(1 - TRAIN_RATIO), random_state=RANDOM_SEED)
    relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val_files, test_files = train_test_split(remaining, test_size=relative_test, random_state=RANDOM_SEED)

    for split_name, split_files in [("train", train_files), ("val", val_files), ("test", test_files)]:
        for f in split_files:
            shutil.copy2(f, PROCESSED_DIR / split_name / label / f.name)

    split_summary.append({"class": label, "train": len(train_files), "val": len(val_files),
                          "test": len(test_files), "total": len(files)})

split_df = pd.DataFrame(split_summary)
print("=" * 55)
print("DATASET SPLIT SUMMARY")
print("=" * 55)
print(split_df.to_string(index=False))
print(f"\nTrain: {split_df['train'].sum()} | Val: {split_df['val'].sum()} | Test: {split_df['test'].sum()} | Total: {split_df['total'].sum()}")

## 6. Save Metadata
CSV tracking every image through the pipeline for full traceability.

In [ ]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)

split_lookup = {}
for split in ["train", "val", "test"]:
    for label in ["fresh", "rotten"]:
        for f in (PROCESSED_DIR / split / label).iterdir():
            split_lookup[f.name] = split

for record in metadata_records:
    fname = record.get("final_name", record["staging_name"])
    record["split"] = split_lookup.get(fname, "removed")

meta_df = pd.DataFrame(metadata_records)
csv_path = METADATA_DIR / "image_metadata.csv"
meta_df.to_csv(csv_path, index=False)

print(f"Metadata saved to: {csv_path}")
print(f"\n--- Distribution per source dataset ---")
print(meta_df.groupby(["source_dataset", "label"]).size().unstack(fill_value=0))
print(f"\n--- Distribution per split ---")
print(meta_df.groupby(["split", "label"]).size().unstack(fill_value=0))
print(f"\n--- Original file formats ---")
print(meta_df["original_format"].value_counts().to_string())

In [ ]:
meta_df.head(10)

## 7. Final Verification
Checking my folder structure, class balance, data leakage, and RGB compliance.

In [ ]:
print("=" * 55)
print("FINAL DATASET STRUCTURE")
print("=" * 55)
for split in ["train", "val", "test"]:
    print(f"\n  {split}/")
    for label in ["fresh", "rotten"]:
        count = len(list((PROCESSED_DIR / split / label).glob(f"*{OUTPUT_FORMAT}")))
        print(f"    {label:8s} → {count} images")

print(f"\n{'=' * 55}")
print("DATA LEAKAGE CHECK")
print("=" * 55)
all_files = {}
leakage_found = False
for split in ["train", "val", "test"]:
    for label in ["fresh", "rotten"]:
        for f in (PROCESSED_DIR / split / label).glob(f"*{OUTPUT_FORMAT}"):
            if f.name in all_files:
                print(f"  ⚠ LEAKAGE: {f.name} in both {all_files[f.name]} and {split}!")
                leakage_found = True
            all_files[f.name] = split
if not leakage_found:
    print("✅ No data leakage detected — all three sets are disjoint.")

print(f"\n{'=' * 55}")
print("CLASS BALANCE")
print("=" * 55)
for split in ["train", "val", "test"]:
    fresh_n = len(list((PROCESSED_DIR / split / "fresh").glob(f"*{OUTPUT_FORMAT}")))
    rotten_n = len(list((PROCESSED_DIR / split / "rotten").glob(f"*{OUTPUT_FORMAT}")))
    total = fresh_n + rotten_n
    if total > 0:
        print(f"  {split:5s}:  fresh {fresh_n:4d} ({fresh_n/total*100:5.1f}%)  |  rotten {rotten_n:4d} ({rotten_n/total*100:5.1f}%)")

print(f"\n{'=' * 55}")
print("RGB VERIFICATION (sample check)")
print("=" * 55)
non_rgb = []
for split in ["train", "val", "test"]:
    for label in ["fresh", "rotten"]:
        for f in list((PROCESSED_DIR / split / label).glob(f"*{OUTPUT_FORMAT}"))[:20]:
            if Image.open(f).mode != "RGB":
                non_rgb.append(f"{split}/{label}/{f.name}")
if non_rgb:
    print(f"⚠ Found {len(non_rgb)} non-RGB images")
else:
    print("✅ All sampled images are RGB (3-channel).")